<a href="https://colab.research.google.com/github/KatreenGhobrial/RepoCloudComputing/blob/main/Tirgulim/Trigul11/Tut11_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AI Agents Tutorial - Cloud Computing Course
# Lecture 11: AI Agents, Agentic RAG, and MCP
# Google Colab Notebook

"""
This tutorial covers:
1. Introduction to AI Agents
2. Different Types of AI Agents (with implementations)
3. Agentic RAG Systems
4. Model Context Protocol (MCP) Basics
5. Practical Examples and Exercises
"""

# ============================================================================
# PART 1: INSTALLATION AND SETUP
# ============================================================================

print("=" * 80)
print("PART 1: Installing Required Libraries")
print("=" * 80)

# Install required packages
!pip install -q openai anthropic
!pip install -q langchain langchain-community langchain-openai
!pip install -q chromadb sentence-transformers
!pip install -q faiss-cpu
!pip install -q numpy pandas matplotlib

print("\n✓ All libraries installed successfully!\n")

# Import necessary libraries
import random
import time
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from enum import Enum
import numpy as np

print("✓ Libraries imported successfully!\n")

In [ ]:

# ============================================================================
# PART 2: SIMPLE REFLEX AGENT
# ============================================================================

print("=" * 80)
print("PART 2: Simple Reflex Agent")
print("=" * 80)

"""
Simple Reflex Agents:
- React to current percepts only
- Follow condition-action rules
- No memory of past events
Example: Thermostat, Traffic Light
"""

class ThermostatAgent:
    """
    A simple reflex agent that controls heating based on temperature.
    """
    def __init__(self, target_temp: float = 22.0):
        self.target_temp = target_temp
        self.heater_on = False

    def perceive_and_act(self, current_temp: float) -> str:
        """React to current temperature reading."""
        if current_temp < self.target_temp and not self.heater_on:
            self.heater_on = True
            return f"🔥 Temperature {current_temp}°C is below target {self.target_temp}°C. Turning heater ON."
        elif current_temp >= self.target_temp and self.heater_on:
            self.heater_on = False
            return f"✓ Temperature {current_temp}°C reached target {self.target_temp}°C. Turning heater OFF."
        else:
            status = "ON" if self.heater_on else "OFF"
            return f"→ Temperature {current_temp}°C. Heater status: {status}. No action needed."

# Demo: Simple Reflex Agent
print("\n--- Thermostat Agent Demo ---")
thermostat = ThermostatAgent(target_temp=22.0)

# Simulate temperature readings
temperatures = [18.0, 19.5, 21.0, 22.0, 22.5, 21.5, 20.0]
for temp in temperatures:
    action = thermostat.perceive_and_act(temp)
    print(action)
    time.sleep(0.5)

In [ ]:

# ============================================================================
# PART 3: MODEL-BASED REFLEX AGENT
# ============================================================================

print("\n" + "=" * 80)
print("PART 3: Model-Based Reflex Agent")
print("=" * 80)

"""
Model-Based Reflex Agents:
- Maintain internal state/model
- Track world changes over time
- Use history to make better decisions
"""

class RobotNavigator:
    """
    A model-based reflex agent that navigates a grid while remembering visited locations.
    """
    def __init__(self, grid_size: int = 5):
        self.grid_size = grid_size
        self.position = [0, 0]  # Start at origin
        self.visited = set()
        self.visited.add(tuple(self.position))
        self.obstacles = set()  # Internal model of obstacles

    def perceive_obstacle(self, obstacle_pos: tuple):
        """Update internal model with obstacle information."""
        self.obstacles.add(obstacle_pos)
        print(f"🚧 Obstacle detected and remembered at {obstacle_pos}")

    def move(self, direction: str) -> str:
        """Move in a direction, considering internal model."""
        moves = {
            'up': [0, 1],
            'down': [0, -1],
            'left': [-1, 0],
            'right': [1, 0]
        }

        if direction not in moves:
            return "Invalid direction"

        # Calculate new position
        new_pos = [
            self.position[0] + moves[direction][0],
            self.position[1] + moves[direction][1]
        ]

        # Check boundaries
        if not (0 <= new_pos[0] < self.grid_size and 0 <= new_pos[1] < self.grid_size):
            return f"❌ Cannot move {direction}: out of bounds"

        # Check obstacles (using internal model)
        if tuple(new_pos) in self.obstacles:
            return f"❌ Cannot move {direction}: obstacle remembered at {tuple(new_pos)}"

        # Move and update state
        self.position = new_pos
        is_new = tuple(self.position) not in self.visited
        self.visited.add(tuple(self.position))

        status = "new location ✨" if is_new else "previously visited"
        return f"→ Moved {direction} to {self.position} ({status})"

# Demo: Model-Based Reflex Agent
print("\n--- Robot Navigator Demo ---")
robot = RobotNavigator(grid_size=5)

# Add some obstacles
robot.perceive_obstacle((1, 1))
robot.perceive_obstacle((2, 0))

# Navigate
commands = ['right', 'right', 'up', 'left', 'left', 'up', 'right']
for cmd in commands:
    result = robot.move(cmd)
    print(result)
    time.sleep(0.5)

print(f"\n📊 Total locations visited: {len(robot.visited)}")
print(f"📍 Final position: {robot.position}")


In [ ]:

# ============================================================================
# PART 4: GOAL-BASED AGENT
# ============================================================================

print("\n" + "=" * 80)
print("PART 4: Goal-Based Agent")
print("=" * 80)

"""
Goal-Based Agents:
- Choose actions to achieve specific goals
- Plan ahead and consider future consequences
- Use reasoning to select optimal path
"""

class PathPlannerAgent:
    """
    A goal-based agent that plans a path to reach a destination.
    """
    def __init__(self, grid_size: int = 5):
        self.grid_size = grid_size
        self.position = [0, 0]
        self.goal = None
        self.obstacles = set()

    def set_goal(self, goal_position: List[int]):
        """Set the goal position."""
        self.goal = goal_position
        print(f"🎯 Goal set to: {self.goal}")

    def add_obstacle(self, obstacle_pos: tuple):
        """Add an obstacle to avoid."""
        self.obstacles.add(obstacle_pos)

    def plan_path(self) -> List[str]:
        """Plan a path to reach the goal using simple A* approach."""
        if not self.goal:
            return []

        # Simple path planning (Manhattan distance heuristic)
        path = []
        current = self.position.copy()

        while current != self.goal:
            # Move horizontally first
            if current[0] < self.goal[0]:
                next_pos = [current[0] + 1, current[1]]
                if tuple(next_pos) not in self.obstacles:
                    path.append('right')
                    current = next_pos
                    continue
            elif current[0] > self.goal[0]:
                next_pos = [current[0] - 1, current[1]]
                if tuple(next_pos) not in self.obstacles:
                    path.append('left')
                    current = next_pos
                    continue

            # Then move vertically
            if current[1] < self.goal[1]:
                next_pos = [current[0], current[1] + 1]
                if tuple(next_pos) not in self.obstacles:
                    path.append('up')
                    current = next_pos
                    continue
            elif current[1] > self.goal[1]:
                next_pos = [current[0], current[1] - 1]
                if tuple(next_pos) not in self.obstacles:
                    path.append('down')
                    current = next_pos
                    continue

            # If stuck, try alternative
            break

        return path

    def execute_plan(self, plan: List[str]) -> bool:
        """Execute the planned path."""
        print(f"\n📋 Executing plan: {' → '.join(plan)}")

        for i, move in enumerate(plan):
            moves = {'up': [0, 1], 'down': [0, -1], 'left': [-1, 0], 'right': [1, 0]}
            self.position[0] += moves[move][0]
            self.position[1] += moves[move][1]
            print(f"  Step {i+1}: Moved {move} to {self.position}")
            time.sleep(0.3)

        reached_goal = self.position == self.goal
        if reached_goal:
            print(f"✓ Goal reached at {self.position}!")
        else:
            print(f"❌ Failed to reach goal. Current position: {self.position}")

        return reached_goal

# Demo: Goal-Based Agent
print("\n--- Path Planner Agent Demo ---")
planner = PathPlannerAgent(grid_size=5)

# Set obstacles
planner.add_obstacle((1, 1))
planner.add_obstacle((2, 1))
print("🚧 Obstacles added at: (1,1) and (2,1)")

# Set goal and plan
planner.set_goal([3, 2])
plan = planner.plan_path()
print(f"\n🗺️  Planned path has {len(plan)} steps")

# Execute
planner.execute_plan(plan)

In [ ]:

# ============================================================================
# PART 5: UTILITY-BASED AGENT
# ============================================================================

print("\n" + "=" * 80)
print("PART 5: Utility-Based Agent")
print("=" * 80)

"""
Utility-Based Agents:
- Maximize utility/performance measure
- Handle conflicting goals
- Consider tradeoffs between multiple objectives
"""

class RouteOptimizerAgent:
    """
    A utility-based agent that optimizes route selection based on multiple factors.
    """
    def __init__(self):
        self.preferences = {
            'time': 0.4,      # Weight for minimizing time
            'cost': 0.3,      # Weight for minimizing cost
            'comfort': 0.3    # Weight for maximizing comfort
        }

    def calculate_utility(self, route: Dict[str, float]) -> float:
        """
        Calculate utility score for a route.
        Higher score = better route
        """
        # Normalize values (assuming ranges: time 0-120 min, cost 0-50, comfort 0-10)
        time_score = 1 - (route['time'] / 120)  # Lower time = higher score
        cost_score = 1 - (route['cost'] / 50)   # Lower cost = higher score
        comfort_score = route['comfort'] / 10    # Higher comfort = higher score

        # Calculate weighted utility
        utility = (
            self.preferences['time'] * time_score +
            self.preferences['cost'] * cost_score +
            self.preferences['comfort'] * comfort_score
        )

        return utility

    def select_best_route(self, routes: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Select the route with highest utility."""
        best_route = None
        best_utility = -1

        print("\n🔍 Evaluating routes:\n")
        for route in routes:
            utility = self.calculate_utility(route)
            route['utility'] = utility
            print(f"  {route['name']:15} | Time: {route['time']:3.0f}min | "
                  f"Cost: ${route['cost']:4.1f} | Comfort: {route['comfort']}/10 | "
                  f"Utility: {utility:.3f}")

            if utility > best_utility:
                best_utility = utility
                best_route = route

        return best_route

# Demo: Utility-Based Agent
print("\n--- Route Optimizer Agent Demo ---")
optimizer = RouteOptimizerAgent()

# Define different route options
routes = [
    {'name': 'Highway',     'time': 30,  'cost': 15.0, 'comfort': 7},
    {'name': 'Scenic Route', 'time': 60,  'cost': 10.0, 'comfort': 9},
    {'name': 'City Streets', 'time': 45,  'cost': 5.0,  'comfort': 4},
    {'name': 'Express',     'time': 25,  'cost': 25.0, 'comfort': 8},
]

best = optimizer.select_best_route(routes)
print(f"\n✓ Best route selected: {best['name']} (Utility: {best['utility']:.3f})")

In [ ]:

# ============================================================================
# PART 6: LEARNING AGENT (Q-Learning Example)
# ============================================================================

print("\n" + "=" * 80)
print("PART 6: Learning Agent")
print("=" * 80)

"""
Learning Agents:
- Improve performance through experience
- Adapt to new environments
- Use reinforcement learning principles
"""

class QLearningAgent:
    """
    A learning agent that uses Q-Learning to learn optimal actions.
    """
    def __init__(self, states: int, actions: int, learning_rate: float = 0.1,
                 discount_factor: float = 0.9, exploration_rate: float = 0.3):
        self.q_table = np.zeros((states, actions))
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = exploration_rate
        self.actions = actions

    def choose_action(self, state: int) -> int:
        """Choose action using epsilon-greedy strategy."""
        if random.random() < self.epsilon:
            return random.randint(0, self.actions - 1)  # Explore
        else:
            return np.argmax(self.q_table[state])  # Exploit

    def learn(self, state: int, action: int, reward: float, next_state: int):
        """Update Q-table based on experience."""
        current_q = self.q_table[state, action]
        max_next_q = np.max(self.q_table[next_state])
        new_q = current_q + self.lr * (reward + self.gamma * max_next_q - current_q)
        self.q_table[state, action] = new_q

    def get_q_table(self):
        """Return current Q-table."""
        return self.q_table

# Demo: Learning Agent in Simple Grid World
print("\n--- Q-Learning Agent Demo ---")
print("Training agent to navigate a 4x4 grid to reach goal...\n")

# Simple 4x4 grid: states 0-15, goal at state 15
# Actions: 0=up, 1=right, 2=down, 3=left
agent = QLearningAgent(states=16, actions=4)

def simulate_step(state, action):
    """Simulate one step in the grid."""
    row, col = state // 4, state % 4

    if action == 0:   # up
        row = max(0, row - 1)
    elif action == 1:  # right
        col = min(3, col + 1)
    elif action == 2:  # down
        row = min(3, row + 1)
    elif action == 3:  # left
        col = max(0, col - 1)

    next_state = row * 4 + col

    # Reward structure
    if next_state == 15:  # Goal
        reward = 100
    elif next_state == state:  # Hit wall
        reward = -1
    else:
        reward = -1  # Step penalty

    return next_state, reward

# Training loop
episodes = 100
for episode in range(episodes):
    state = 0  # Start at top-left
    total_reward = 0
    steps = 0

    while state != 15 and steps < 50:  # Until goal or max steps
        action = agent.choose_action(state)
        next_state, reward = simulate_step(state, action)
        agent.learn(state, action, reward, next_state)

        state = next_state
        total_reward += reward
        steps += 1

    if (episode + 1) % 20 == 0:
        print(f"Episode {episode + 1}: Total Reward = {total_reward:.1f}, Steps = {steps}")

print("\n✓ Training complete!")
print("\nLearned Q-values for state 0 (top-left):")
print(f"  Up: {agent.q_table[0, 0]:.2f}")
print(f"  Right: {agent.q_table[0, 1]:.2f}")
print(f"  Down: {agent.q_table[0, 2]:.2f}")
print(f"  Left: {agent.q_table[0, 3]:.2f}")


In [ ]:
# ============================================================================
# PART 7: AGENTIC RAG SYSTEM
# ============================================================================

print("\n" + "=" * 80)
print("PART 8: Agentic RAG System")
print("=" * 80)

"""
Agentic RAG:
- Incorporates AI agents into RAG systems
- Multiple specialized agents for different data sources
- Autonomous decision-making about what to retrieve
"""

class AgenticRAG:
    """
    An agentic RAG system with multiple specialized retrieval agents.
    """
    def __init__(self):
        self.agents = {}

    def register_agent(self, agent_name: str, knowledge_base: List[str]):
        """Register a specialized retrieval agent."""
        self.agents[agent_name] = {
            'knowledge_base': knowledge_base,
            'queries_handled': 0
        }
        print(f"✓ Registered agent: {agent_name}")

    def route_query(self, query: str) -> str:
        """Route query to most appropriate agent."""
        query_lower = query.lower()

        # Simple routing logic based on keywords
        if any(word in query_lower for word in ['cloud', 'scalability', 'deployment']):
            return 'cloud_expert'
        elif any(word in query_lower for word in ['rag', 'retrieval', 'llm']):
            return 'rag_expert'
        elif any(word in query_lower for word in ['agent', 'autonomous', 'learning']):
            return 'agent_expert'
        else:
            return 'cloud_expert'  # Default

    def query(self, query: str) -> Dict[str, Any]:
        """Process query through appropriate agent."""
        agent_name = self.route_query(query)

        if agent_name not in self.agents:
            return {'error': 'No suitable agent found'}

        agent = self.agents[agent_name]
        agent['queries_handled'] += 1

        # Retrieve relevant information
        results = self._retrieve_from_agent(query, agent['knowledge_base'])

        return {
            'agent': agent_name,
            'query': query,
            'results': results,
            'confidence': 0.85
        }

    def _retrieve_from_agent(self, query: str, knowledge_base: List[str]) -> List[str]:
        """Retrieve information from agent's knowledge base."""
        query_words = set(query.lower().split())
        scored_docs = []

        for doc in knowledge_base:
            doc_words = set(doc.lower().split())
            intersection = query_words.intersection(doc_words)
            union = query_words.union(doc_words)
            similarity = len(intersection) / len(union) if union else 0
            if similarity > 0:
                scored_docs.append((similarity, doc))

        scored_docs.sort(reverse=True)
        return [doc for _, doc in scored_docs[:2]]

    def get_stats(self):
        """Get statistics about agent usage."""
        stats = {}
        for name, agent in self.agents.items():
            stats[name] = agent['queries_handled']
        return stats

# Demo: Agentic RAG System
print("\n--- Agentic RAG System Demo ---\n")
agentic_rag = AgenticRAG()

# Register specialized agents
agentic_rag.register_agent('cloud_expert', [
    "Cloud computing provides scalability, resource optimization, and cost efficiency for AI workloads.",
    "Cloud platforms offer serverless execution, global accessibility, and integration with various services.",
    "AI agents in cloud can dynamically allocate resources based on computational demands.",
])

agentic_rag.register_agent('rag_expert', [
    "RAG retrieves facts from external knowledge bases to ground LLM responses in accurate information.",
    "RAG systems combine retrieval mechanisms with generative language models for better answers.",
    "Agentic RAG uses multiple specialized agents to query different types of data sources.",
])

agentic_rag.register_agent('agent_expert', [
    "AI agents are autonomous software entities that perceive, decide, and act to achieve goals.",
    "Learning agents improve through experience using reinforcement learning and adaptation.",
    "Goal-based agents plan ahead and consider future consequences when making decisions.",
])

# Test queries
test_queries = [
    "How does cloud computing support AI agents?",
    "What is RAG and how does it work?",
    "Explain learning agents and their components",
]

for query in test_queries:
    print(f"\n{'='*70}")
    result = agentic_rag.query(query)
    print(f"🤔 Query: {result['query']}")
    print(f"🎯 Routed to: {result['agent']}")
    print(f"📚 Retrieved information:")
    for i, info in enumerate(result['results'], 1):
        print(f"   {i}. {info}")
    print(f"💯 Confidence: {result['confidence']}")

# Show usage statistics
print(f"\n{'='*70}")
print("📊 Agent Usage Statistics:")
stats = agentic_rag.get_stats()
for agent, count in stats.items():
    print(f"   {agent}: {count} queries handled")


In [ ]:

#----------------------
#       Tirgul Answer
#---------------------

class BasilSimpleAgent:
   def preceive_and_action(self,temperature,humidity,soil):
       if temperature > 35:
          return "Temperature is high. Action: move basil to cooler place"
       elif humidity > 80:
          return "Humidity is high. Action: check for fungal diease risk."
       elif soil < 30:
           return "Soil moisture is low. Action: water the basil."
       else:
          return "Basil conditions are normal. No action needed"

basil_agent = BasilSimpleAgent()

print(basil_agent.preceive_and_action(30,60,50))
print(basil_agent.preceive_and_action(25,85,50))
print(basil_agent.preceive_and_action(25,60,20))



print("We choose the simple Relex Agent because it respods directly to current sensor values with simple condition action rules")

In [ ]:

# ============================================================================
# BONUS: MCP BASICS (Conceptual Implementation)
# ============================================================================

print("\n" + "=" * 80)
print("PART 9: Model Context Protocol (MCP) Basics")
print("=" * 80)

"""
MCP (Model Context Protocol):
- Standardizes how AI assistants communicate with external resources
- Provides secure, controlled access to data sources
- Enables seamless integration across different tools
"""

class MCPServer:
    """
    A simplified MCP server that provides resources and tools.
    """
    def __init__(self, name: str):
        self.name = name
        self.resources = {}
        self.tools = {}

    def register_resource(self, resource_name: str, data: Any):
        """Register a read-only resource."""
        self.resources[resource_name] = data
        print(f"✓ Resource registered: {resource_name}")

    def register_tool(self, tool_name: str, function):
        """Register an interactive tool."""
        self.tools[tool_name] = function
        print(f"✓ Tool registered: {tool_name}")

    def get_resource(self, resource_name: str) -> Any:
        """Retrieve a resource."""
        return self.resources.get(resource_name)

    def execute_tool(self, tool_name: str, **kwargs) -> Any:
        """Execute a tool with given parameters."""
        if tool_name in self.tools:
            return self.tools[tool_name](**kwargs)
        return None

class MCPClient:
    """
    A simplified MCP client that connects to MCP servers.
    """
    def __init__(self, name: str):
        self.name = name
        self.connections = {}

    def connect(self, server: MCPServer):
        """Connect to an MCP server."""
        self.connections[server.name] = server
        print(f"✓ Connected to MCP server: {server.name}")

    def read_resource(self, server_name: str, resource_name: str) -> Any:
        """Read a resource from connected server."""
        if server_name in self.connections:
            return self.connections[server_name].get_resource(resource_name)
        return None

    def call_tool(self, server_name: str, tool_name: str, **kwargs) -> Any:
        """Call a tool on connected server."""
        if server_name in self.connections:
            return self.connections[server_name].execute_tool(tool_name, **kwargs)
        return None

# Demo: MCP System
print("\n--- MCP System Demo ---\n")

# Create MCP Server
file_server = MCPServer("FileSystemServer")

# Register resources (read-only data)
file_server.register_resource("config", {
    "api_key": "sk-demo-12345",
    "max_tokens": 1000,
    "temperature": 0.7
})

file_server.register_resource("logs", [
    "2025-01-03 10:00:00 - System started",
    "2025-01-03 10:05:00 - Agent initialized",
    "2025-01-03 10:10:00 - Processing request"
])

# Register tools (interactive capabilities)
def search_files(query: str) -> List[str]:
    """Search for files matching query."""
    mock_files = ["config.json", "data.csv", "model.pkl", "report.pdf"]
    return [f for f in mock_files if query.lower() in f.lower()]

def count_lines(filename: str) -> int:
    """Count lines in a file."""
    mock_counts = {"config.json": 42, "data.csv": 1500, "report.pdf": 250}
    return mock_counts.get(filename, 0)

file_server.register_tool("search", search_files)
file_server.register_tool("count_lines", count_lines)

# Create MCP Client (AI Application)
ai_client = MCPClient("AI_Assistant")
ai_client.connect(file_server)

# Use MCP to access resources and tools
print("\n1️⃣ Reading configuration resource:")
config = ai_client.read_resource("FileSystemServer", "config")
print(f"   Config: {json.dumps(config, indent=2)}")

print("\n2️⃣ Reading logs resource:")
logs = ai_client.read_resource("FileSystemServer", "logs")
for log in logs:
    print(f"   {log}")

print("\n3️⃣ Using search tool:")
results = ai_client.call_tool("FileSystemServer", "search", query="data")
print(f"   Search results for 'data': {results}")

print("\n4️⃣ Using count_lines tool:")
count = ai_client.call_tool("FileSystemServer", "count_lines", filename="data.csv")
print(f"   Line count for 'data.csv': {count}")


In [ ]:

# ============================================================================
# EXERCISES AND CHALLENGES
# ============================================================================

print("\n" + "=" * 80)
print("PART 10: Exercises for Students")
print("=" * 80)

print("""
EXERCISES:

1. SIMPLE REFLEX AGENT:
   Create a traffic light agent that changes signals based on traffic sensor inputs.

2. MODEL-BASED AGENT:
   Extend the RobotNavigator to remember the shortest path to previously visited locations.

3. GOAL-BASED AGENT:
   Implement A* algorithm for the PathPlannerAgent to find optimal paths around obstacles.

4. UTILITY-BASED AGENT:
   Create a restaurant recommendation agent that considers price, distance, rating, and cuisine.

5. LEARNING AGENT:
   Train a Q-Learning agent to play a simple game (e.g., Tic-Tac-Toe).

6. RAG SYSTEM:
   Build a RAG system that can answer questions about your course materials.

7. AGENTIC RAG:
   Create an agentic RAG with 3 specialized agents for different subject domains.

8. MCP INTEGRATION:
   Design an MCP server that provides access to a database and implements CRUD operations.

BONUS CHALLENGE:
Combine all concepts to create a cloud-based AI assistant that:
- Uses multiple agent types
- Implements RAG for knowledge retrieval
- Connects to external resources via MCP
- Learns from user interactions

""")

# ============================================================================
# SUMMARY AND RESOURCES
# ============================================================================

print("=" * 80)
print("SUMMARY")
print("=" * 80)

print("""
✓ Simple Reflex Agents: React to current state only
✓ Model-Based Agents: Maintain internal state and history
✓ Goal-Based Agents: Plan actions to achieve specific goals
✓ Utility-Based Agents: Optimize decisions with multiple objectives
✓ Learning Agents: Improve through experience and adaptation

✓ RAG: Retrieval-Augmented Generation for grounded responses
✓ Agentic RAG: Multiple specialized agents for different data sources
✓ MCP: Standardized protocol for AI-external resource communication

KEY TAKEAWAYS:
1. AI agents provide autonomous, intelligent automation
2. Cloud computing enables scalable deployment of agents
3. Different agent types suit different problems
4. RAG grounds LLM responses in factual data
5. MCP standardizes integration with external systems

RESOURCES:
- IBM AI Agent Types: https://www.ibm.com/think/topics/ai-agent-types
- Anthropic MCP Documentation: https://docs.anthropic.com/
- OpenAI API Documentation: https://platform.openai.com/docs
- LangChain Documentation: https://python.langchain.com/

""")

print("=" * 80)
print("Tutorial Complete! Happy Learning! 🎓")
print("=" * 80)